# Four-class H&E tissue segmentation


In [ ]:
from __future__ import annotations
from pathlib import Path
import os
import sys

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "config" / "project.yaml").is_file()
             and (path / "src" / "pathology").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter within this repository.")
sys.path.insert(0, str(ROOT / "src"))
from pathology.config import load_paths
PATHS = load_paths(ROOT)
os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")
os.environ.setdefault("HF_HOME", str(ROOT / ".cache" / "huggingface"))
os.environ.setdefault("HF_MODULES_CACHE", str(ROOT / ".cache" / "huggingface" / "modules"))
os.environ.setdefault("TORCH_HOME", str(ROOT / ".cache" / "torch"))


## Imports


In [ ]:
import gc
import importlib
import importlib.util
import json
import math
import os
import re
import sys
import time
import traceback
import types
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Optional
import cv2
import numpy as np
import pandas as pd
import tifffile
import torch
import zarr
from numcodecs import Blosc
from PIL import Image
from scipy import ndimage as ndi
from torch import nn
if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()


## Configuration


In [ ]:
@dataclass
class Config:
    HE_DIR: str = str(PATHS.he_dir)
    REGISTRATION_RESULTS_DIR: str = str(PATHS.registration)
    TIGER_SEG_ROOT: str = str(ROOT / 'vendor' / 'tiger')
    CHECKPOINT: str = str(PATHS.segmentation_checkpoint)
    OUTPUT_DIR: str = str(PATHS.segmentation)
    RUN_MODE: str = 'full'
    RUN_CASE_IDS: tuple[str, ...] = ()
    TARGET_MPP: float = 0.5
    FALLBACK_BASE_MPP: float = 0.25
    PATCH_SIZE: int = 512
    STRIDE: int = 256
    TISSUE_MIN_FRACTION: float = 0.05
    THUMBNAIL_LONG_EDGE: int = 1800
    BATCH_SIZE_GPU: int = 4
    BATCH_SIZE_CPU: int = 1
    CLASS_NAMES: tuple[str, ...] = ('tumour', 'tumour_associated_stroma', 'inflamed_stroma', 'other')
    CLASS_COLORS_RGB: tuple[tuple[int, int, int], ...] = ((255, 0, 0), (180, 120, 0), (255, 255, 0), (160, 160, 160))
    IGNORE_VALUE: int = 255
    WRITE_BIGTIFF: bool = True
    RANDOM_SEED: int = 6211
    REQUIRE_CUDA: bool = True
    USE_AMP: bool = True
    SKIP_COMPLETED: bool = True
    CONTINUE_ON_ERROR: bool = True
CFG = Config()
assert CFG.RUN_MODE == 'full'
assert CFG.PATCH_SIZE == 512, 'The provided model was trained with 512x512 patches.'
assert 0 < CFG.STRIDE <= CFG.PATCH_SIZE
HE_DIR = Path(CFG.HE_DIR)
REG_DIR = Path(CFG.REGISTRATION_RESULTS_DIR)
SEG_ROOT = Path(CFG.TIGER_SEG_ROOT)
CHECKPOINT = Path(CFG.CHECKPOINT)
OUTPUT_DIR = Path(CFG.OUTPUT_DIR)
for path, label in [(HE_DIR, 'H&E directory'), (REG_DIR, 'registration results'), (SEG_ROOT, 'TIGER segmentation code'), (CHECKPOINT, 'checkpoint')]:
    if not path.exists():
        raise FileNotFoundError(f'Missing {label}: {path}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(CFG.RANDOM_SEED)
torch.manual_seed(CFG.RANDOM_SEED)
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if CFG.REQUIRE_CUDA and DEVICE.type != 'cuda':
    raise RuntimeError('CUDA is required. In Jupyter choose Kernel > Change kernel > a CUDA-enabled Python environment, then restart and run all cells.')
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True
BATCH_SIZE = CFG.BATCH_SIZE_GPU if DEVICE.type == 'cuda' else CFG.BATCH_SIZE_CPU


## QPTIFF reader


In [ ]:
def _yx_shape(shape: tuple[int, ...], axes: str) -> tuple[int, int]:
    axes = axes.upper()
    if 'Y' not in axes or 'X' not in axes:
        raise ValueError(f'No Y/X axes: axes={axes}, shape={shape}')
    return (int(shape[axes.index('Y')]), int(shape[axes.index('X')]))

def _select_main_series(tif: tifffile.TiffFile) -> int:
    candidates = []
    for index, series in enumerate(tif.series):
        try:
            h, w = _yx_shape(tuple(series.shape), series.axes)
            candidates.append((h * w, index))
        except ValueError:
            pass
    if not candidates:
        raise ValueError('No TIFF series contains X/Y axes')
    return max(candidates)[1]

def _ratio(tag_value) -> float:
    if isinstance(tag_value, tuple):
        return float(tag_value[0]) / max(float(tag_value[1]), 1e-12)
    return float(tag_value)

def base_mpp_from_page(page: tifffile.TiffPage, fallback: float) -> tuple[float, str]:
    x_tag = page.tags.get('XResolution')
    unit_tag = page.tags.get('ResolutionUnit')
    if x_tag is not None and unit_tag is not None:
        pixels_per_unit = _ratio(x_tag.value)
        unit_text = str(unit_tag.value).lower()
        if pixels_per_unit > 0 and ('centimeter' in unit_text or str(unit_tag.value) == '3'):
            return (10000.0 / pixels_per_unit, 'TIFF XResolution cm')
        if pixels_per_unit > 0 and ('inch' in unit_text or str(unit_tag.value) == '2'):
            return (25400.0 / pixels_per_unit, 'TIFF XResolution inch')
    return (float(fallback), 'fallback')

def inspect_qptiff(path: Path, target_mpp: float=CFG.TARGET_MPP) -> dict[str, Any]:
    with tifffile.TiffFile(path) as tif:
        series_index = _select_main_series(tif)
        series = tif.series[series_index]
        levels = list(getattr(series, 'levels', [series]))
        full_h, full_w = _yx_shape(tuple(levels[0].shape), levels[0].axes)
        base_mpp, mpp_source = base_mpp_from_page(levels[0].pages[0], CFG.FALLBACK_BASE_MPP)
        rows = []
        for index, level in enumerate(levels):
            h, w = _yx_shape(tuple(level.shape), level.axes)
            downsample = float(np.mean([full_h / h, full_w / w]))
            rows.append({'level': index, 'height': h, 'width': w, 'axes': level.axes, 'shape': tuple((int(v) for v in level.shape)), 'downsample': downsample, 'mpp': base_mpp * downsample})
    selected = min(rows, key=lambda row: abs(math.log(max(row['mpp'], 1e-09) / target_mpp)))
    return {'path': str(path), 'name': path.name, 'series_index': series_index, 'full_height': full_h, 'full_width': full_w, 'base_mpp': base_mpp, 'mpp_source': mpp_source, 'levels': rows, 'selected_level': selected['level'], 'selected_height': selected['height'], 'selected_width': selected['width'], 'selected_mpp': selected['mpp'], 'selected_downsample': selected['downsample']}

def _array_to_rgb(array: np.ndarray, axes: str) -> np.ndarray:
    arr = np.asarray(array)
    labels = list(axes.upper())
    for index in range(arr.ndim - 1, -1, -1):
        if labels[index] not in {'Y', 'X', 'C', 'S'}:
            arr = np.take(arr, 0, axis=index)
            labels.pop(index)
    channel = 'C' if 'C' in labels else 'S' if 'S' in labels else None
    if channel is None:
        arr = np.moveaxis(arr, (labels.index('Y'), labels.index('X')), (0, 1))
        arr = np.repeat(arr[..., None], 3, axis=2)
    else:
        arr = np.moveaxis(arr, (labels.index('Y'), labels.index('X'), labels.index(channel)), (0, 1, 2))
        if arr.shape[2] == 1:
            arr = np.repeat(arr, 3, axis=2)
        elif arr.shape[2] == 2:
            arr = np.concatenate([arr, arr[..., :1]], axis=2)
        else:
            arr = arr[..., :3]
    if arr.dtype != np.uint8:
        arr = np.clip(arr, 0, 255).astype(np.uint8)
    return np.ascontiguousarray(arr)

class QPTiffLevelReader:

    def __init__(self, path: Path, target_mpp: float=CFG.TARGET_MPP):
        self.path = Path(path)
        self.meta = inspect_qptiff(self.path, target_mpp)
        self.level = int(self.meta['selected_level'])
        with tifffile.TiffFile(self.path) as tif:
            level = list(getattr(tif.series[self.meta['series_index']], 'levels', []))[self.level]
            self.axes = level.axes
        self.store = tifffile.imread(self.path, series=self.meta['series_index'], level=self.level, aszarr=True)
        self.array = zarr.open(self.store, mode='r')
        self.height = int(self.meta['selected_height'])
        self.width = int(self.meta['selected_width'])

    def read_patch(self, x: int, y: int, size: int) -> np.ndarray:
        x, y = (int(x), int(y))
        read_w = max(0, min(size, self.width - x))
        read_h = max(0, min(size, self.height - y))
        labels = list(self.axes.upper())
        selection = []
        for label in labels:
            if label == 'Y':
                selection.append(slice(y, y + read_h))
            elif label == 'X':
                selection.append(slice(x, x + read_w))
            elif label in {'C', 'S'}:
                selection.append(slice(None))
            else:
                selection.append(0)
        patch = _array_to_rgb(np.asarray(self.array[tuple(selection)]), self.axes)
        if patch.shape[:2] != (size, size):
            canvas = np.full((size, size, 3), 255, np.uint8)
            canvas[:patch.shape[0], :patch.shape[1]] = patch
            patch = canvas
        return patch

    def close(self):
        close = getattr(self.store, 'close', None)
        if callable(close):
            close()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.close()

def read_thumbnail(path: Path, long_edge: int=CFG.THUMBNAIL_LONG_EDGE) -> tuple[np.ndarray, dict]:
    meta = inspect_qptiff(path)
    level = min(meta['levels'], key=lambda row: abs(math.log(max(row['height'], row['width']) / long_edge)))
    with tifffile.TiffFile(path) as tif:
        series = tif.series[meta['series_index']]
        source = list(getattr(series, 'levels', [series]))[level['level']]
        rgb = _array_to_rgb(source.asarray(), source.axes)
    return (rgb, {**meta, 'thumbnail_level': level['level']})


## Slide inventory


In [ ]:
def extract_case_id(path: Path) -> str:
    match = re.match('^(?P<case>.+?)_Scan\\d+$', path.stem, flags=re.IGNORECASE)
    return match.group('case') if match else path.stem
he_files = sorted([*HE_DIR.glob('*.qptiff'), *HE_DIR.glob('*.qptif')], key=lambda p: p.name.lower())
if not he_files:
    raise RuntimeError(f'No H&E QPTIFF files found under {HE_DIR}')
he_by_case = {extract_case_id(path): path for path in he_files}
if len(he_by_case) != len(he_files):
    raise RuntimeError('Duplicate H&E case identifiers detected')
case_ids = sorted(he_by_case)
if CFG.RUN_CASE_IDS:
    missing = sorted(set(CFG.RUN_CASE_IDS) - set(case_ids))
    if missing:
        raise ValueError(f'Unknown RUN_CASE_IDS: {missing}')
    case_ids = [case for case in case_ids if case in set(CFG.RUN_CASE_IDS)]
inventory_rows = []
for case_id in case_ids:
    meta = inspect_qptiff(he_by_case[case_id])
    package = REG_DIR / 'pairs' / f'{case_id}__mIF_to_HE' / 'v5_transform_package.npz'
    inventory_rows.append({'case_id': case_id, 'he_path': str(he_by_case[case_id]), 'full_width': meta['full_width'], 'full_height': meta['full_height'], 'base_mpp': meta['base_mpp'], 'mpp_source': meta['mpp_source'], 'selected_level': meta['selected_level'], 'selected_mpp': meta['selected_mpp'], 'selected_width': meta['selected_width'], 'selected_height': meta['selected_height'], 'selected_downsample': meta['selected_downsample'], 'registration_transform_package': str(package) if package.exists() else '', 'registration_transform_exists': package.exists()})
inventory_df = pd.DataFrame(inventory_rows)
inventory_df.to_csv(OUTPUT_DIR / 'segmentation_inventory_v2.csv', index=False, encoding='utf-8-sig')
display(inventory_df)
assert len(case_ids) == len(he_by_case), 'V2 is configured to process all discovered cases.'


## Tissue mask and patch grid


In [ ]:
def make_tissue_mask(rgb: np.ndarray) -> np.ndarray:
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    saturation = hsv[..., 1]
    mask = (saturation > 12) & (gray < 245) | (gray < 190)
    mask = ndi.binary_closing(mask, iterations=3)
    mask = ndi.binary_opening(mask, iterations=1)
    mask = ndi.binary_fill_holes(mask)
    labels, count = ndi.label(mask)
    if count:
        sizes = np.bincount(labels.ravel())
        minimum = max(64, int(mask.size * 0.0002))
        keep = sizes >= minimum
        keep[0] = False
        mask = keep[labels]
    return mask.astype(bool)

def axis_positions(length: int, patch: int, stride: int) -> list[int]:
    if length <= patch:
        return [0]
    positions = list(range(0, length - patch + 1, stride))
    last = length - patch
    if positions[-1] != last:
        positions.append(last)
    return positions

def tissue_fraction_for_patch(tissue_mask: np.ndarray, x: int, y: int, patch: int, level_w: int, level_h: int) -> float:
    th, tw = tissue_mask.shape
    x0 = max(0, min(tw - 1, int(math.floor(x * tw / level_w))))
    y0 = max(0, min(th - 1, int(math.floor(y * th / level_h))))
    x1 = max(x0 + 1, min(tw, int(math.ceil((x + patch) * tw / level_w))))
    y1 = max(y0 + 1, min(th, int(math.ceil((y + patch) * th / level_h))))
    return float(tissue_mask[y0:y1, x0:x1].mean())

def build_patch_grid(path: Path) -> tuple[list[dict], np.ndarray, np.ndarray, dict]:
    thumb, meta = read_thumbnail(path)
    tissue = make_tissue_mask(thumb)
    width, height = (int(meta['selected_width']), int(meta['selected_height']))
    rows = []
    for y in axis_positions(height, CFG.PATCH_SIZE, CFG.STRIDE):
        for x in axis_positions(width, CFG.PATCH_SIZE, CFG.STRIDE):
            fraction = tissue_fraction_for_patch(tissue, x, y, CFG.PATCH_SIZE, width, height)
            if fraction >= CFG.TISSUE_MIN_FRACTION:
                rows.append({'x': x, 'y': y, 'tissue_fraction': fraction})
    if not rows:
        raise RuntimeError(f'No tissue patches detected for {path.name}')
    return (rows, thumb, tissue, meta)

def spatial_smoke_subset(rows: list[dict], count: int) -> list[dict]:
    ordered = sorted(rows, key=lambda row: (row['y'], row['x']))
    count = min(max(1, count), len(ordered))
    indices = np.unique(np.linspace(0, len(ordered) - 1, count).round().astype(int))
    return [ordered[index] for index in indices]

def save_grid_preview(case_id: str, thumb: np.ndarray, tissue: np.ndarray, rows: list[dict], meta: dict):
    canvas = thumb.copy()
    h, w = canvas.shape[:2]
    for row in rows:
        x0 = round(row['x'] * w / meta['selected_width'])
        y0 = round(row['y'] * h / meta['selected_height'])
        x1 = round((row['x'] + CFG.PATCH_SIZE) * w / meta['selected_width'])
        y1 = round((row['y'] + CFG.PATCH_SIZE) * h / meta['selected_height'])
        cv2.rectangle(canvas, (x0, y0), (x1, y1), (0, 255, 255), 1)
    tissue_rgb = np.repeat((tissue.astype(np.uint8) * 255)[..., None], 3, axis=2)
    case_dir = OUTPUT_DIR / case_id
    case_dir.mkdir(parents=True, exist_ok=True)
    Image.fromarray(canvas).save(case_dir / 'patch_grid_preview.jpg', quality=90)
    Image.fromarray(tissue_rgb).save(case_dir / 'tissue_mask_thumbnail.png')
    return canvas


## Checkpoint loading


In [ ]:
class LegacyActivation(nn.Module):

    def __init__(self, name=None, **params):
        super().__init__()
        if name in (None, 'identity'):
            self.activation = nn.Identity(**params)
        elif name == 'softmax2d':
            self.activation = nn.Softmax(dim=1, **params)
        elif name == 'softmax':
            self.activation = nn.Softmax(**params)
        elif name == 'sigmoid':
            self.activation = nn.Sigmoid()
        else:
            raise ValueError(f'Unsupported legacy activation: {name}')

    def forward(self, x):
        return self.activation(x)

def legacy_drop_path(x, drop_prob=0.0, training=False):
    if drop_prob == 0.0 or not training:
        return x
    keep_prob = 1.0 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)
    random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
    random_tensor.floor_()
    return x.div(keep_prob) * random_tensor

class LegacyDropPath(nn.Module):

    def __init__(self, drop_prob=None):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        return legacy_drop_path(x, self.drop_prob, self.training)

def register_checkpoint_compatibility(seg_root: Path):
    legacy_drop = types.ModuleType('timm.models.layers.drop')
    legacy_drop.drop_path = legacy_drop_path
    legacy_drop.DropPath = LegacyDropPath
    sys.modules['timm.models.layers.drop'] = legacy_drop
    if importlib.util.find_spec('segmentation_models_pytorch') is None:
        smp = types.ModuleType('segmentation_models_pytorch')
        base = types.ModuleType('segmentation_models_pytorch.base')
        modules = types.ModuleType('segmentation_models_pytorch.base.modules')
        modules.Activation = LegacyActivation
        base.modules = modules
        smp.base = base
        sys.modules['segmentation_models_pytorch'] = smp
        sys.modules['segmentation_models_pytorch.base'] = base
        sys.modules['segmentation_models_pytorch.base.modules'] = modules
    for name in ['model.net.van.net', 'model.net.van', 'model.net', 'model']:
        sys.modules.pop(name, None)
    model_pkg = types.ModuleType('model')
    model_pkg.__path__ = [str(seg_root / 'model')]
    net_pkg = types.ModuleType('model.net')
    net_pkg.__path__ = [str(seg_root / 'model' / 'net')]
    van_pkg = types.ModuleType('model.net.van')
    van_pkg.__path__ = [str(seg_root / 'model' / 'net' / 'van')]
    sys.modules['model'] = model_pkg
    sys.modules['model.net'] = net_pkg
    sys.modules['model.net.van'] = van_pkg
    importlib.import_module('model.net.van.net')
register_checkpoint_compatibility(SEG_ROOT)
started = time.time()
MODEL = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
MODEL = MODEL.to(DEVICE).eval()
for module in MODEL.modules():
    if module.__class__.__name__ == 'DropPath' and (not hasattr(module, 'scale_by_keep')):
        module.scale_by_keep = True
with torch.inference_mode():
    test_output = MODEL(torch.zeros((1, 3, 512, 512), device=DEVICE))
assert tuple(test_output.shape) == (1, 4, 512, 512)
assert torch.allclose(test_output.sum(dim=1).mean(), torch.tensor(1.0, device=DEVICE), atol=0.0001)
del test_output


## Patch inference


In [ ]:
IMAGENET_MEAN = np.asarray([0.485, 0.456, 0.406], np.float32).reshape(1, 1, 3)
IMAGENET_STD = np.asarray([0.229, 0.224, 0.225], np.float32).reshape(1, 1, 3)
CLASS_COLORS = np.asarray(CFG.CLASS_COLORS_RGB, np.uint8)

def preprocess_patch(rgb: np.ndarray) -> torch.Tensor:
    normalized = (rgb.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(np.ascontiguousarray(normalized.transpose(2, 0, 1)))

def infer_rgb_patches(patches: list[np.ndarray]) -> np.ndarray:
    batch = torch.stack([preprocess_patch(patch) for patch in patches]).to(DEVICE, non_blocking=DEVICE.type == 'cuda')
    use_amp = bool(CFG.USE_AMP and DEVICE.type == 'cuda')
    with torch.inference_mode(), torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
        probabilities = MODEL(batch)
    result = probabilities.detach().float().cpu().numpy()
    del batch, probabilities
    return result

def colorize_classes(labels: np.ndarray) -> np.ndarray:
    ignored = labels == CFG.IGNORE_VALUE
    safe = np.where(ignored, 0, labels).astype(np.int64)
    rgb = CLASS_COLORS[safe]
    rgb[ignored] = 0
    return rgb

def make_overlay(rgb: np.ndarray, labels: np.ndarray, alpha: float=0.45) -> np.ndarray:
    colors = colorize_classes(labels)
    valid = labels != CFG.IGNORE_VALUE
    output = rgb.astype(np.float32).copy()
    output[valid] = (1 - alpha) * output[valid] + alpha * colors[valid]
    return np.clip(output, 0, 255).astype(np.uint8)

def gaussian_kernel(size: int, steep: float=6.0) -> np.ndarray:
    x = cv2.getGaussianKernel(size, size / steep)
    x /= np.average(x)
    kernel = x @ x.T
    return kernel.astype(np.float32)

def class_fraction_dict(labels: np.ndarray) -> dict[str, float]:
    total = float(labels.size)
    return {f'fraction_{name}': float((labels == index).sum() / total) for index, name in enumerate(CFG.CLASS_NAMES)}


## Resumable slide reconstruction


In [ ]:
def export_mask_bigtiff(mask, output_path: Path, tumour_only: bool=False, tile: int=512):
    height, width = map(int, mask.shape)

    def tiles():
        for y in range(0, height, tile):
            for x in range(0, width, tile):
                block = np.asarray(mask[y:min(y + tile, height), x:min(x + tile, width)])
                if tumour_only:
                    block = (block == 0).astype(np.uint8) * 255
                if block.shape != (tile, tile):
                    padded = np.full((tile, tile), 0 if tumour_only else CFG.IGNORE_VALUE, np.uint8)
                    padded[:block.shape[0], :block.shape[1]] = block
                    block = padded
                yield block
    with tifffile.TiffWriter(output_path, bigtiff=True) as writer:
        writer.write(data=tiles(), shape=(height, width), dtype=np.uint8, tile=(tile, tile), compression='deflate', photometric='minisblack', metadata={'axes': 'YX'})

def save_full_qc(case_dir: Path, thumb: np.ndarray, mask) -> None:
    height, width = map(int, mask.shape)
    step = max(1, math.ceil(max(height, width) / CFG.THUMBNAIL_LONG_EDGE))
    small_mask = np.asarray(mask[::step, ::step])
    mask_rgb = colorize_classes(small_mask)
    he_small = cv2.resize(thumb, (small_mask.shape[1], small_mask.shape[0]), interpolation=cv2.INTER_AREA)
    overlay = make_overlay(he_small, small_mask)
    Image.fromarray(mask_rgb).save(case_dir / 'class_mask_thumbnail.png')
    Image.fromarray(overlay).save(case_dir / 'segmentation_overlay_thumbnail.jpg', quality=92)
    Image.fromarray((small_mask == 0).astype(np.uint8) * 255).save(case_dir / 'tumour_bulk_thumbnail.png')

def run_full_case(case_id: str, path: Path) -> dict[str, Any]:
    started = time.time()
    rows, thumb, tissue, meta = build_patch_grid(path)
    save_grid_preview(case_id, thumb, tissue, rows, meta)
    case_dir = OUTPUT_DIR / case_id
    mask_path = case_dir / 'class_mask_level.zarr'
    mask = zarr.open(str(mask_path), mode='w', shape=(meta['selected_height'], meta['selected_width']), chunks=(512, 512), dtype='u1', fill_value=CFG.IGNORE_VALUE, compressor=Blosc(cname='zstd', clevel=5, shuffle=Blosc.BITSHUFFLE))
    grouped = defaultdict(list)
    for row in rows:
        grouped[int(row['y'])].append(row)
    y_positions = axis_positions(meta['selected_height'], CFG.PATCH_SIZE, CFG.STRIDE)
    width = int(meta['selected_width'])
    tissue_x_index = np.minimum(tissue.shape[1] - 1, ((np.arange(width) + 0.5) * tissue.shape[1] / width).astype(np.int64))
    buffer_gb = CFG.PATCH_SIZE * width * 5 * 4 / 1024 ** 3
    if buffer_gb > 1.5:
        raise MemoryError(f'Rolling fusion buffer would require {buffer_gb:.2f} GB')
    scores = np.zeros((CFG.PATCH_SIZE, width, 4), np.float32)
    weights = np.zeros((CFG.PATCH_SIZE, width), np.float32)
    kernel = gaussian_kernel(CFG.PATCH_SIZE)
    buffer_top = 0
    inferred = 0

    def flush(delta: int):
        nonlocal buffer_top, scores, weights
        if delta <= 0:
            return
        count = min(delta, CFG.PATCH_SIZE, meta['selected_height'] - buffer_top)
        if count > 0:
            tissue_y_index = np.minimum(tissue.shape[0] - 1, ((np.arange(buffer_top, buffer_top + count) + 0.5) * tissue.shape[0] / meta['selected_height']).astype(np.int64))
            tissue_valid = tissue[np.ix_(tissue_y_index, tissue_x_index)]
            valid = (weights[:count] > 0) & tissue_valid
            labels = scores[:count].argmax(axis=2).astype(np.uint8)
            labels[~valid] = CFG.IGNORE_VALUE
            mask[buffer_top:buffer_top + count, :] = labels
        shift = min(delta, CFG.PATCH_SIZE)
        if shift < CFG.PATCH_SIZE:
            scores[:-shift] = scores[shift:]
            weights[:-shift] = weights[shift:]
            scores[-shift:] = 0
            weights[-shift:] = 0
        else:
            scores.fill(0)
            weights.fill(0)
        buffer_top += delta
    with QPTiffLevelReader(path) as reader:
        for row_index, y in enumerate(y_positions):
            if y > buffer_top:
                flush(y - buffer_top)
            row_items = sorted(grouped.get(y, []), key=lambda row: row['x'])
            for start in range(0, len(row_items), BATCH_SIZE):
                batch_rows = row_items[start:start + BATCH_SIZE]
                patches = [reader.read_patch(row['x'], y, CFG.PATCH_SIZE) for row in batch_rows]
                probabilities = infer_rgb_patches(patches)
                for row, probability in zip(batch_rows, probabilities):
                    x = int(row['x'])
                    scores[:, x:x + CFG.PATCH_SIZE, :] += probability.transpose(1, 2, 0) * kernel[..., None]
                    weights[:, x:x + CFG.PATCH_SIZE] += kernel
                    inferred += 1
    flush(meta['selected_height'] - buffer_top)
    save_full_qc(case_dir, thumb, mask)
    if CFG.WRITE_BIGTIFF:
        export_mask_bigtiff(mask, case_dir / 'four_class_mask_0p5mpp.tif', tumour_only=False)
        export_mask_bigtiff(mask, case_dir / 'tumour_bulk_mask_0p5mpp.tif', tumour_only=True)
    registration_package = REG_DIR / 'pairs' / f'{case_id}__mIF_to_HE' / 'v5_transform_package.npz'
    result = {'case_id': case_id, 'mode': 'full', 'status': 'ok', 'he_path': str(path), 'torch_version': torch.__version__, 'cuda_runtime': torch.version.cuda, 'gpu_name': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '', 'batch_size': BATCH_SIZE, 'amp_enabled': bool(CFG.USE_AMP and DEVICE.type == 'cuda'), 'selected_level': meta['selected_level'], 'selected_mpp': meta['selected_mpp'], 'selected_width': meta['selected_width'], 'selected_height': meta['selected_height'], 'selected_downsample': meta['selected_downsample'], 'candidate_tissue_patches': len(rows), 'inferred_patches': inferred, 'class_mask_zarr': str(mask_path), 'registration_transform_package': str(registration_package) if registration_package.exists() else '', 'device': str(DEVICE), 'checkpoint': str(CHECKPOINT), 'elapsed_seconds': time.time() - started, 'output_dir': str(case_dir)}
    (case_dir / 'full_run_metadata.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    return result

def read_completed_result(case_id: str) -> Optional[dict[str, Any]]:
    metadata_path = OUTPUT_DIR / case_id / 'full_run_metadata.json'
    if not metadata_path.exists():
        return None
    try:
        result = json.loads(metadata_path.read_text(encoding='utf-8'))
        mask_path = Path(result.get('class_mask_zarr', ''))
        if result.get('status') == 'ok' and mask_path.exists():
            return result
    except Exception:
        return None
    return None

def write_running_summary(results: list[dict[str, Any]]) -> pd.DataFrame:
    dataframe = pd.DataFrame(results)
    destination = OUTPUT_DIR / 'segmentation_summary_v2.csv'
    temporary = OUTPUT_DIR / 'segmentation_summary_v2.tmp.csv'
    dataframe.to_csv(temporary, index=False, encoding='utf-8-sig')
    os.replace(temporary, destination)
    return dataframe
full_results = []
if CFG.RUN_MODE == 'full':
    overall_started = time.time()
    for case_number, case_id in enumerate(case_ids, start=1):
        completed = read_completed_result(case_id) if CFG.SKIP_COMPLETED else None
        if completed is not None:
            completed = {**completed, 'resume_action': 'skipped_completed'}
            full_results.append(completed)
            write_running_summary(full_results)
            continue
        try:
            result = run_full_case(case_id, he_by_case[case_id])
            result['resume_action'] = 'processed_now'
            full_results.append(result)
        except Exception as error:
            failure = {'case_id': case_id, 'mode': 'full', 'status': 'failed', 'error_type': type(error).__name__, 'error': str(error), 'traceback': traceback.format_exc(), 'device': str(DEVICE), 'checkpoint': str(CHECKPOINT), 'resume_action': 'failed_now'}
            case_dir = OUTPUT_DIR / case_id
            case_dir.mkdir(parents=True, exist_ok=True)
            (case_dir / 'full_run_failure.json').write_text(json.dumps(failure, ensure_ascii=False, indent=2), encoding='utf-8')
            full_results.append(failure)
            if not CFG.CONTINUE_ON_ERROR:
                write_running_summary(full_results)
                raise
        finally:
            write_running_summary(full_results)
            gc.collect()
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()
        elapsed = time.time() - overall_started
    full_df = write_running_summary(full_results)
    display(full_df)


## Run summary


In [ ]:
full_df[['case_id', 'status', 'inferred_patches', 'elapsed_seconds']]
